### Human-in-the-Loop with LangGraph Middleware

This example builds on the tool-calling agent from `1-BasicChatBot` and adds **approval gates** before selected tools run.

Key ideas:
- `HumanInTheLoopMiddleware` pauses the agent when the LLM requests a protected tool call
- A **checkpointer** (here `InMemorySaver`) persists graph state while waiting for a human decision
- You resume with `Command(resume={"decisions": [...]})` and the same `thread_id`

Allowed human decisions per tool:
- `approve` — run the tool as proposed
- `edit` — change tool name/args before running
- `reject` — skip execution and tell the model why
- `respond` — answer on behalf of the tool (no execution)

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b")

### Tools

We reuse the same custom `multiply` tool from the basic chatbot notebook.

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int

    Returns:
        int: multiplication of a and b
    """
    return a * b

### Agent + HumanInTheLoopMiddleware

`create_agent` compiles a LangGraph agent under the hood. The middleware runs **after the model** and intercepts tool calls listed in `interrupt_on`.

In [ ]:
agent = create_agent(
    model=llm,
    tools=[multiply],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require human approval before multiply runs
                "multiply": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "description": "Review this multiplication before the agent executes it.",
                }
            },
            description_prefix="Tool execution pending approval",
        )
    ],
    # Required for interrupts — use a persistent saver in production
    checkpointer=InMemorySaver(),
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

### Step 1 — Run until interrupt

The graph stops when the model requests `multiply`. Inspect `result["__interrupt__"]` for the pending action.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is 6 times 7? Use the multiply tool."}]},
    config,
)

if "__interrupt__" in result:
    hitl_request = result["__interrupt__"][0].value
    print("Pending actions:")
    for action in hitl_request["action_requests"]:
        print(f"  - {action['name']}({action['args']})")
        print(f"    {action.get('description', '')}")
else:
    print(result["messages"][-1].content)

### Step 2 — Resume with a human decision

Pick one of the patterns below. Use the **same** `config` / `thread_id`.

In [ ]:
# Option A: approve the proposed tool call
resume_value = {"decisions": [{"type": "approve"}]}

final = agent.invoke(Command(resume=resume_value), config)
print(final["messages"][-1].content)

In [ ]:
# Option B: edit args before execution (re-run Step 1 first in a fresh thread)
# resume_value = {
#     "decisions": [
#         {
#             "type": "edit",
#             "edited_action": {"name": "multiply", "args": {"a": 6, "b": 8}},
#         }
#     ]
# }
# final = agent.invoke(Command(resume=resume_value), config)

In [ ]:
# Option C: reject with a reason
# resume_value = {
#     "decisions": [
#         {
#             "type": "reject",
#             "message": "Do not run multiply; answer from memory instead.",
#         }
#     ]
# }
# final = agent.invoke(Command(resume=resume_value), config)

### Helper — run until complete

In a real UI you would loop: run → show interrupt → collect decision → resume.

In [ ]:
def run_with_auto_approve(agent, user_message: str, config: dict):
    result = agent.invoke({"messages": [{"role": "user", "content": user_message}]}, config)

    while "__interrupt__" in result:
        hitl_request = result["__interrupt__"][0].value
        print("Auto-approving:", hitl_request["action_requests"][0]["name"])
        result = agent.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config,
        )

    return result["messages"][-1].content


# run_with_auto_approve(agent, "What is 9 times 11?", config)

### Visualize the compiled graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    pass